# Final four optimized models

Logistic Regression, Decision Tree, Random Forest and Gradient Boosting. Each family retains its strongest eligible validation variant. Training/test accuracy is reported at its frozen threshold. All models pass the current fitting screens; this does not guarantee absence of overfitting or underfitting.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
import joblib
root=Path.cwd()
if root.name=='notebooks': root=root.parent
sys.path.insert(0,str(root/'src'))
folder=root/'outputs/final_four'
selection=json.loads((folder/'selection.json').read_text())
print('Selected model:',selection['model'])
summary=pd.read_csv(folder/'accuracy_summary.csv')
display(summary.style.format({c:'{:.2%}' for c in summary.columns if 'accuracy' in c or 'gap' in c}))

Selected model: random_forest


,Model,Source,Threshold,Validation accuracy,CV accuracy,Threshold-matched CV gap,Training accuracy,Test accuracy,Train-test gap,Selected,Fit assessment
0,logistic_regression,optimized_logistic_regression,0.425000,68.29%,69.18%,0.81%,68.67%,67.37%,1.30%,False,Passes current screens; not proof of zero fitting risk
1,decision_tree,decision_tree,0.425000,66.10%,68.22%,1.16%,65.28%,65.69%,-0.41%,False,Passes current screens; not proof of zero fitting risk
2,random_forest,random_forest,0.425000,68.63%,69.59%,4.15%,71.56%,67.12%,4.43%,True,Passes current screens; not proof of zero fitting risk
3,gradient_boosting,gradient_boosting,0.375000,67.88%,69.54%,2.40%,67.21%,65.88%,1.33%,False,Passes current screens; not proof of zero fitting risk


In [2]:
from model_workflow import paths,load_features,matrix,TARGET,metrics
import numpy as np
data,fingerprint=load_features(paths(root)['processed'])
assert fingerprint==selection['fingerprint']
for _,row in summary.iterrows():
    bundle=joblib.load(folder/f'{row.Model}.joblib')
    for split,column in [('train','Training accuracy'),('validation','Validation accuracy'),('test','Test accuracy')]:
        part=data[data.split.eq(split)]
        probabilities=bundle['pipeline'].predict_proba(matrix(part))[:,1]
        accuracy=metrics(row.Model,part[TARGET],probabilities,bundle['metadata']['Threshold'])['Accuracy']
        assert np.isclose(accuracy,row[column])
print('Verified all four saved models against reported train, validation and test accuracy.')

Verified all four saved models against reported train, validation and test accuracy.


In [3]:
selected=joblib.load(folder/f"{selection['model']}.joblib")
print('Prediction threshold:',selected['metadata']['Threshold'])
print(selection['limitations'])

Prediction threshold: 0.425
Validation and test were previously inspected. Scores are reused-holdout results; no guarantee of zero fitting risk.


Optimization uses fold-fitted preprocessing, regularization and customer-disjoint cross-validation. Fitting screens limit mean training–CV and training–validation gaps to 5 percentage points and require CV improvement over the majority baseline. Older experiments are historical; use only this notebook and `outputs/final_four` for the final four-model comparison.